# TACS CIFAR-100 trên Google Colab

Notebook này chạy trọn quy trình DINOv3/TACS trên CIFAR-100, lưu checkpoint vào Google Drive và tự resume sau khi Colab ngắt phiên. Quy trình chỉ được coi là hoàn tất khi mọi run có `metrics.json`, kết quả test và ba file `multiseed_summary.*`. Cấu hình v2 bắt đầu sạch với batch hiệu dụng 64, warm-up 5 epoch và learning rate ổn định hơn cho DINOv3.

Trước khi chạy: chọn **Runtime → Change runtime type → GPU**, upload `TACS_Colab_Bundle.zip` vào thư mục gốc My Drive, rồi chạy các cell theo thứ tự.

In [ ]:
from pathlib import Path

# Chỉ cần sửa các giá trị này khi chuyển seed hoặc muốn đổi cấu hình.
BUNDLE_PATH = Path('/content/drive/MyDrive/TACS_Colab_Bundle.zip')
PROJECT_DIR = Path('/content/BTL_XLA')
DRIVE_OUTPUTS_ROOT = Path('/content/drive/MyDrive/TACS_outputs_v2')
SEEDS = [42, 43, 44]
BASELINES = ['no_context', 'random', 'similarity', 'tacs', 'two_stage']
EPOCHS = 100
BATCH_SIZE = 4  # Micro-batch vừa VRAM T4.
GRADIENT_ACCUMULATION_STEPS = 16  # Batch hiệu dụng = 4 * 16 = 64.
LEARNING_RATE = 1e-4
WARMUP_EPOCHS = 5
NUM_WORKERS = 2
USE_BUNDLED_CHECKPOINT = False  # Không dùng checkpoint pilot có recipe cũ.
print('Configured seeds:', SEEDS)
print('Effective batch size:', BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print('Drive outputs:', DRIVE_OUTPUTS_ROOT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert BUNDLE_PATH.is_file(), f'Không tìm thấy bundle: {BUNDLE_PATH}'
print('Bundle size (MB):', round(BUNDLE_PATH.stat().st_size / 1024**2, 1))

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

# Source chạy trên SSD tạm của Colab; chỉ output/checkpoint đặt trên Drive.
os.chdir('/content')  # Cho phép chạy lại cell dù cwd đang nằm trong PROJECT_DIR.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    archive.extractall(PROJECT_DIR)

# Không để pip cài lại PyTorch/CUDA của Colab. Cài dependency nhỏ
# trước, sau đó cài project editable với --no-deps. Không dùng -q để
# nếu có lỗi thì log hiển thị đầy đủ.
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        '--disable-pip-version-check',
        'setuptools>=68', 'wheel', 'timm>=1.0.20', 'psutil>=5.9',
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        '--disable-pip-version-check', '--no-deps',
        '-e', str(PROJECT_DIR),
    ],
    check=True,
)
os.chdir(PROJECT_DIR)
source_dir = str(PROJECT_DIR / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import importlib
importlib.invalidate_caches()
import tacs
print('Installed project from:', PROJECT_DIR)
print('TACS import:', tacs.__file__)

In [ ]:
# Mặc định v2 không nạp checkpoint pilot cũ vì recipe optimizer đã thay đổi.
bootstrap_root = PROJECT_DIR / 'outputs'
copied = []
if USE_BUNDLED_CHECKPOINT and bootstrap_root.is_dir():
    for source in bootstrap_root.rglob('*'):
        if not source.is_file():
            continue
        destination = DRIVE_OUTPUTS_ROOT / source.relative_to(bootstrap_root)
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            copied.append(str(destination))
print('Bundled checkpoint enabled:', USE_BUNDLED_CHECKPOINT)
print(f'Bootstrap files copied: {len(copied)}')
for path in copied:
    print(' -', path)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Colab chưa bật GPU: Runtime → Change runtime type → GPU'
gpu = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('GPU:', gpu.name)
print('VRAM (GB):', round(gpu.total_memory / 1024**3, 1))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Kiểm tra tiến độ hiện có trên Drive trước khi train.
import json

experiment_root = DRIVE_OUTPUTS_ROOT / 'cifar100/dinov3_vits16'
for seed in SEEDS:
    for baseline in BASELINES:
        run_dir = experiment_root / f'seed_{seed}' / baseline
        metrics_path = run_dir / 'metrics.json'
        latest_path = run_dir / 'checkpoint_latest.pt'
        if metrics_path.is_file():
            metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
            test_result = metrics.get('test_result') or {}
            print(
                f'[COMPLETE] seed={seed} {baseline}: '
                f"test_accuracy={test_result.get('accuracy', 'N/A')}"
            )
        elif latest_path.is_file():
            checkpoint = torch.load(
                latest_path, map_location='cpu', weights_only=True
            )
            print(
                f"[RESUME]   seed={seed} {baseline}: "
                f"epoch={int(checkpoint['epoch']) + 1}/{EPOCHS}"
            )
            del checkpoint
        else:
            print(f'[PENDING]  seed={seed} {baseline}')

In [ ]:
# HF_TOKEN là tùy chọn; thêm trong Colab Secrets để tải model ổn định hơn.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN enabled')
else:
    print('HF_TOKEN not set; public download will still be attempted')

In [ ]:
# CIFAR-100 được tải vào SSD tạm mỗi runtime; split seed 42 nằm trong bundle.
from tacs.data import build_cifar100
datasets = build_cifar100(
    root=PROJECT_DIR / 'data',
    split_file=PROJECT_DIR / 'data/splits/cifar100_seed42.json',
    download=True,
)
print('CIFAR-100 ready:', {
    'query_train': len(datasets.query_train),
    'candidate_pool': len(datasets.candidate_pool),
    'validation': len(datasets.validation),
    'test': len(datasets.test),
})

In [ ]:
# Xem lệnh trước khi train.
base_command = [
    sys.executable, '-m', 'tacs.multiseed',
    '--dataset', 'cifar100',
    '--backbone', 'dinov3_vits16',
    '--device', 'cuda',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--lr', str(LEARNING_RATE),
    '--warmup-epochs', str(WARMUP_EPOCHS),
    '--num-workers', str(NUM_WORKERS),
    '--data-root', str(PROJECT_DIR / 'data'),
    '--split-file', str(PROJECT_DIR / 'data/splits/cifar100_seed42.json'),
    '--outputs-root', str(DRIVE_OUTPUTS_ROOT),
    '--seeds', *map(str, SEEDS),
    '--baselines', *BASELINES,
]
subprocess.run([*base_command, '--plan-only'], check=True)

In [ ]:
# Cell train chính. Có thể chạy lại sau khi runtime bị ngắt; trainer
# tự bỏ qua run đã xong và resume run còn dở từ checkpoint_latest.pt.
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    base_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, base_command)
print('Training command completed. Run the verification cell below.')

In [ ]:
# Xác minh hoàn tất. Cell này cố ý báo lỗi nếu còn thiếu run/file.
from tacs.multiseed import RunnerConfig, is_complete_run, write_summaries

verification_config = RunnerConfig(
    dataset='cifar100',
    backbone='dinov3_vits16',
    seeds=tuple(SEEDS),
    baselines=tuple(BASELINES),
    outputs_root=DRIVE_OUTPUTS_ROOT,
    data_root=PROJECT_DIR / 'data',
    split_file=PROJECT_DIR / 'data/splits/cifar100_seed42.json',
    device='cuda',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    lr=LEARNING_RATE,
    warmup_epochs=WARMUP_EPOCHS,
    num_workers=NUM_WORKERS,
)

missing_runs = []
for seed in SEEDS:
    for baseline in BASELINES:
        complete = is_complete_run(verification_config, seed, baseline)
        print(('[OK]      ' if complete else '[MISSING] '), f'seed={seed} {baseline}')
        if not complete:
            missing_runs.append((seed, baseline))

assert not missing_runs, (
    'Thí nghiệm chưa hoàn tất. Hãy chạy lại cell train để resume: '
    f'{missing_runs}'
)

summary_paths = write_summaries(verification_config)
for path in summary_paths:
    assert path.is_file(), f'Thiếu summary: {path}'
    print('[SUMMARY]', path)

print('\n===== FINAL SUMMARY =====')
print(summary_paths[2].read_text(encoding='utf-8'))
print('TẤT CẢ THÍ NGHIỆM ĐÃ HOÀN TẤT.')

## Khi Colab bị ngắt

Kết nối lại GPU và chạy notebook từ đầu. Source/dataset trên `/content` sẽ được tạo lại, nhưng checkpoint trong `MyDrive/TACS_outputs_v2` được giữ nguyên. Runner tự bỏ qua run có `metrics.json` hợp lệ và tiếp tục run còn dở.

Chỉ coi toàn bộ thí nghiệm là xong khi cell xác minh in `TẤT CẢ THÍ NGHIỆM ĐÃ HOÀN TẤT.` và Drive có `multiseed_summary.json`, `multiseed_summary.csv`, `multiseed_summary.md`.